# MinutesAI — Google Colab GPU inference server

This notebook runs the project's AI models on a Colab GPU and exposes a temporary HTTPS endpoint. Your laptop runs the React UI and a small FastAPI proxy; audio is forwarded to this notebook for Whisper transcription and Llama 3.2 summarization.

**Before starting**
1. In Colab choose **Runtime → Change runtime type → T4 GPU** (if available).
2. Add a Colab Secret named `HF_TOKEN` containing your Hugging Face access token (🔑 Secrets panel).
3. Make sure your Hugging Face account has accepted access to `meta-llama/Llama-3.2-3B-Instruct`.
4. Run the cells from top to bottom. Keep this notebook/runtime running while using the app.

The public tunnel URL and API key are temporary. Never share them publicly or commit them to GitHub.


In [ ]:
!pip -q install --upgrade transformers==4.57.6 accelerate bitsandbytes fastapi uvicorn python-multipart huggingface_hub

In [ ]:
import os, io, time, secrets, threading, subprocess, re, requests
from pathlib import Path
import torch
from fastapi import FastAPI, UploadFile, File, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata
import uvicorn

assert torch.cuda.is_available(), "No GPU detected. In Colab select Runtime → Change runtime type → T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
# Hugging Face authentication. Add HF_TOKEN in the Colab Secrets panel first.
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a Colab Secret named HF_TOKEN, then rerun this cell.")
login(token=hf_token, add_to_git_credential=False)

WHISPER_MODEL = "openai/whisper-medium.en"
LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
API_KEY = secrets.token_urlsafe(32)
print("Hugging Face authentication complete.")

In [ ]:
# Load Whisper — same checkpoint as the supplied Week 3 Day 5 notebook.
asr = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    torch_dtype=torch.float16,
    device=0,
    token=hf_token,
    return_timestamps=True,
)
print("Whisper is ready.")

In [ ]:
# Load Llama 3.2 3B Instruct using 4-bit NF4 quantization, matching the supplied notebook.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    device_map="auto",
    quantization_config=quant_config,
    token=hf_token,
)
model.eval()
print("Llama is ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Async job-based inference API — avoids Cloudflare 524 timeouts
# ---------------------------------------------------------------------------
import json as _json
import uuid
import logging
from collections import OrderedDict

_log = logging.getLogger("minutesai")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(name)s] %(message)s")

PARSER_VERSION = "3.1-regex-fallback"

app = FastAPI(title="MinutesAI Colab GPU Inference")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

def check_key(x_api_key: str | None):
    if not x_api_key or not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Invalid API key")

# --- In-memory job store (bounded to 50 jobs, oldest evicted) ---
MAX_JOBS = 50

class JobStore:
    def __init__(self, maxsize: int = MAX_JOBS):
        self._jobs: OrderedDict[str, dict] = OrderedDict()
        self._maxsize = maxsize
        self._lock = threading.Lock()

    def create(self, job_id: str) -> dict:
        job = {
            "job_id": job_id,
            "status": "queued",
            "stage": "queued",
            "result": None,
            "error": None,
            "created_at": time.time(),
        }
        with self._lock:
            if len(self._jobs) >= self._maxsize:
                self._jobs.popitem(last=False)
            self._jobs[job_id] = job
        return job

    def get(self, job_id: str) -> dict | None:
        with self._lock:
            return self._jobs.get(job_id)

    def update(self, job_id: str, **kwargs):
        with self._lock:
            if job_id in self._jobs:
                self._jobs[job_id].update(kwargs)

jobs = JobStore()

# --- GPU processing lock (one job at a time on a single T4) ---
_gpu_lock = threading.Lock()

# --- JSON sanitizer: fixes literal control chars inside JSON strings ---
def _sanitize_json_string(text: str) -> str:
    """Fix literal newlines/tabs/control chars inside JSON string values."""
    result = []
    in_string = False
    i = 0
    while i < len(text):
        ch = text[i]
        if in_string:
            if ch == '\\' and i + 1 < len(text):
                result.append(ch)
                result.append(text[i + 1])
                i += 2
                continue
            elif ch == '"':
                in_string = False
                result.append(ch)
            elif ch == '\n':
                result.append('\\n')
            elif ch == '\r':
                result.append('\\r')
            elif ch == '\t':
                result.append('\\t')
            elif ord(ch) < 0x20:
                result.append('\\u{:04x}'.format(ord(ch)))
            else:
                result.append(ch)
        else:
            if ch == '"':
                in_string = True
            result.append(ch)
        i += 1
    return ''.join(result)


def _extract_json_field(text, key):
    """Extract a simple string value for a given JSON key using regex."""
    pattern = r'"' + re.escape(key) + r'"\s*:\s*"'
    match = re.search(pattern, text)
    if not match:
        return None
    start = match.end()
    i = start
    while i < len(text):
        ch = text[i]
        if ch == '\\' and i + 1 < len(text):
            i += 2
            continue
        if ch == '"':
            rest = text[i + 1:].lstrip()
            if rest and rest[0] in (',', '}', ']'):
                return text[start:i]
        i += 1
    return text[start:]


def _extract_json_array(text, key):
    """Extract an array of strings for a given JSON key using regex."""
    pattern = r'"' + re.escape(key) + r'"\s*:\s*\['
    match = re.search(pattern, text)
    if not match:
        return None
    start = match.end()
    depth = 1
    i = start
    in_str = False
    while i < len(text) and depth > 0:
        ch = text[i]
        if in_str:
            if ch == '\\' and i + 1 < len(text):
                i += 2
                continue
            if ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '[':
                depth += 1
            elif ch == ']':
                depth -= 1
        i += 1
    if depth != 0:
        return None
    array_text = text[start:i - 1]
    items = re.findall(r'"((?:[^"\\]|\\.)*)"', array_text)
    return items


def _extract_action_items(text):
    """Extract action_items array of objects from raw text."""
    pattern = r'"action_items"\s*:\s*\['
    match = re.search(pattern, text)
    if not match:
        return None
    start = match.end()
    depth = 1
    i = start
    in_str = False
    while i < len(text) and depth > 0:
        ch = text[i]
        if in_str:
            if ch == '\\' and i + 1 < len(text):
                i += 2
                continue
            if ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '[':
                depth += 1
            elif ch == ']':
                depth -= 1
        i += 1
    if depth != 0:
        return None
    array_text = text[start:i - 1]
    items = []
    obj_pattern = re.compile(r'\{[^{}]*\}')
    for obj_match in obj_pattern.finditer(array_text):
        obj_str = obj_match.group()
        try:
            obj = _json.loads(obj_str)
            items.append(obj)
        except (_json.JSONDecodeError, ValueError):
            task = re.search(r'"task"\s*:\s*"((?:[^"\\]|\\.)*)"', obj_str)
            owner = re.search(r'"owner"\s*:\s*"((?:[^"\\]|\\.)*)"', obj_str)
            due = re.search(r'"due_date"\s*:\s*"((?:[^"\\]|\\.)*)"', obj_str)
            items.append({
                "task": task.group(1) if task else "",
                "owner": owner.group(1) if owner else "Not specified",
                "due_date": due.group(1) if due else "Not specified",
            })
    return items if items else None


def _fallback_regex_parse(text):
    """Last-resort parser: extract fields individually using regex."""
    _log.info("_parse_minutes: Attempting regex-based field extraction")
    data = {}
    for key in ("title", "date", "location", "summary"):
        val = _extract_json_field(text, key)
        if val is not None:
            val = val.replace("\\n", "\n").replace("\\r", "\r").replace("\\t", "\t")
            val = val.replace('\\"', '"').replace("\\\\", "\\")
            data[key] = val
    for key in ("attendees", "discussion_points", "takeaways"):
        val = _extract_json_array(text, key)
        if val is not None:
            data[key] = val
    action_items = _extract_action_items(text)
    if action_items is not None:
        data["action_items"] = action_items
    md = _extract_json_field(text, "markdown")
    if md is not None:
        md = md.replace("\\n", "\n").replace("\\r", "\r").replace("\\t", "\t")
        md = md.replace('\\"', '"').replace("\\\\", "\\")
        data["markdown"] = md
    if data:
        _log.info("_parse_minutes: Regex extraction found keys: %s", list(data.keys()))
    return data


# --- Minutes parser with three-pass parsing + logging ---
def _parse_minutes(raw: str) -> dict:
    """Parse the LLM JSON response into a structured minutes dict."""
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    data = {}
    if start >= 0 and end > start:
        json_text = text[start : end + 1]
        # Pass 1: try raw
        try:
            data = _json.loads(json_text)
            _log.info("_parse_minutes: JSON parsed on first try")
        except (_json.JSONDecodeError, ValueError) as e1:
            _log.warning("_parse_minutes: First json.loads failed: %s", e1)
            # Pass 2: sanitize literal control chars and retry
            try:
                sanitized = _sanitize_json_string(json_text)
                data = _json.loads(sanitized)
                _log.info("_parse_minutes: JSON parsed after sanitizing")
            except (_json.JSONDecodeError, ValueError) as e2:
                _log.warning("_parse_minutes: Second json.loads failed: %s", e2)
                # Pass 3: regex-based field extraction
                data = _fallback_regex_parse(json_text)
    else:
        _log.warning("_parse_minutes: No JSON object braces found in output")

    # Normalize action_items
    raw_actions = data.get("action_items") or []
    actions = []
    for item in raw_actions:
        if isinstance(item, dict):
            actions.append({
                "task": str(item.get("task", "")),
                "owner": str(item.get("owner", "Not specified")),
                "due_date": str(item.get("due_date", "Not specified")),
            })
        else:
            actions.append({"task": str(item), "owner": "Not specified", "due_date": "Not specified"})

    summary = str(data.get("summary") or "")
    markdown = str(data.get("markdown") or "")

    if not data:
        summary = "The model generated a report, but it could not be parsed into structured fields."
        markdown = raw

    result = {
        "title": str(data.get("title") or "Meeting Minutes"),
        "date": str(data.get("date") or "Not specified"),
        "location": str(data.get("location") or "Not specified"),
        "attendees": [str(x) for x in (data.get("attendees") or [])],
        "summary": summary,
        "discussion_points": [str(x) for x in (data.get("discussion_points") or [])],
        "takeaways": [str(x) for x in (data.get("takeaways") or [])],
        "action_items": actions,
        "markdown": markdown,
    }

    _log.info("_parse_minutes result: title=%s, date=%s, summary_len=%d, dp=%d, tk=%d, ai=%d",
              result["title"], result["date"], len(result["summary"]),
              len(result["discussion_points"]), len(result["takeaways"]), len(result["action_items"]))

    return result

# --- Background worker function ---
def _process_job(job_id: str, audio_bytes: bytes, suffix: str):
    temp_path = None
    try:
        jobs.update(job_id, status="processing", stage="transcribing")
        with open(f"/tmp/minutesai_{job_id}{suffix}", "wb") as f:
            f.write(audio_bytes)
            temp_path = f.name

        with _gpu_lock:
            # Whisper transcription
            transcription = asr(temp_path)
            transcript = (transcription.get("text") or "").strip()
            if not transcript:
                jobs.update(job_id, status="failed", error="Whisper did not detect speech in the audio.")
                return

            jobs.update(job_id, stage="summarizing")

            # Llama summarization
            system_message = (
                "You produce meeting minutes from transcripts. "
                "Return ONLY valid JSON — no markdown fences, no commentary. "
                "The JSON object must have exactly these keys:\n"
                "  title (string), date (string), location (string), "
                "attendees (array of strings), "
                "summary (string — a concise executive summary, NOT the full report), "
                "discussion_points (array of strings), "
                "takeaways (array of strings), "
                "action_items (array of objects with keys: task, owner, due_date), "
                "markdown (string — the complete formatted meeting report in Markdown).\n"
                "Use 'Not specified' for any detail not present in the transcript. "
                "Do not invent facts that are absent from the transcript."
            )
            user_prompt = (
                "Create meeting minutes as a single JSON object from the following "
                "transcript. Include all required fields. Do not wrap the JSON in "
                "markdown code fences.\n\nTRANSCRIPT:\n" + transcript
            )
            messages = [
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_prompt},
            ]
            inputs = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to("cuda")
            with torch.inference_mode():
                outputs = model.generate(
                    inputs,
                    max_new_tokens=2000,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            raw = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

        _log.info("Job %s: Raw LLM output length=%d, first 200 chars: %s", job_id, len(raw), repr(raw[:200]))

        minutes = _parse_minutes(raw)
        result = {
            "transcript": transcript,
            "minutes": minutes,
            "transcription_model": WHISPER_MODEL,
            "summarization_model": LLAMA_MODEL,
        }
        jobs.update(job_id, status="completed", stage="completed", result=result)
    except Exception as exc:
        _log.exception("Job %s failed: %s", job_id, exc)
        jobs.update(job_id, status="failed", error=str(exc))
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

# --- Endpoints ---
@app.get("/health")
def health(x_api_key: str | None = Header(default=None)):
    check_key(x_api_key)
    return {
        "status": "ok",
        "gpu": torch.cuda.get_device_name(0),
        "whisper_model": WHISPER_MODEL,
        "llama_model": LLAMA_MODEL,
        "parser_version": PARSER_VERSION,
    }

@app.post("/submit")
async def submit_audio(audio: UploadFile = File(...), x_api_key: str | None = Header(default=None)):
    """Accept audio, return a job_id instantly, process in the background."""
    check_key(x_api_key)
    allowed = {".mp3", ".wav", ".m4a", ".mp4", ".mpeg", ".mpga", ".webm", ".ogg", ".flac"}
    suffix = Path(audio.filename or "").suffix.lower()
    if suffix not in allowed:
        raise HTTPException(status_code=415, detail="Unsupported audio format.")
    data = await audio.read()
    if not data:
        raise HTTPException(status_code=400, detail="Empty audio file.")
    if len(data) > 200 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Audio exceeds 200 MB.")

    job_id = uuid.uuid4().hex[:12]
    jobs.create(job_id)
    worker = threading.Thread(target=_process_job, args=(job_id, data, suffix), daemon=True)
    worker.start()
    return {"job_id": job_id}

@app.get("/status/{job_id}")
def job_status(job_id: str, x_api_key: str | None = Header(default=None)):
    """Return the current status and processing stage of a job."""
    check_key(x_api_key)
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found.")
    return {"job_id": job_id, "status": job["status"], "stage": job["stage"], "error": job["error"]}

@app.get("/result/{job_id}")
def job_result(job_id: str, x_api_key: str | None = Header(default=None)):
    """Return the structured meeting minutes once processing is complete."""
    check_key(x_api_key)
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found.")
    if job["status"] == "failed":
        raise HTTPException(status_code=422, detail=job["error"] or "Processing failed.")
    if job["status"] != "completed":
        raise HTTPException(status_code=202, detail="Job is still processing.")
    return job["result"]


In [ ]:
# Start updated FastAPI on port 8001
def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8001,
        log_level="warning"
    )

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:8001/docs", timeout=2)
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("FastAPI did not start.")

print("Updated MinutesAI API running on port 8001.")

In [ ]:
# Start a temporary Cloudflare HTTPS tunnel (no Cloudflare account required).
cloudflared_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared_path):
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    open(cloudflared_path, "wb").write(r.content)
    os.chmod(cloudflared_path, 0o755)

tunnel = subprocess.Popen(
    [cloudflared_path, "tunnel", "--url", "http://127.0.0.1:8001", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError("Cloudflare tunnel exited. Rerun this cell.")
if not public_url:
    raise RuntimeError("Could not obtain a tunnel URL. Rerun this cell or check the cell output.")

print("\n" + "="*68)
print("COPY THESE VALUES INTO backend/.env ON YOUR LAPTOP")
print("="*68)
print("COLAB_INFERENCE_URL=" + public_url)
print("COLAB_API_KEY=" + API_KEY)
print("="*68)
print("Keep this Colab runtime running while using MinutesAI.")

## Connect your laptop

1. Copy the two values printed in the previous cell.
2. On your laptop, open `minutesai-colab-connected/backend/.env` (create it by copying `.env.example`).
3. Paste the values after `COLAB_INFERENCE_URL=` and `COLAB_API_KEY=`.
4. Start the local FastAPI proxy and React frontend using the README instructions.
5. The dashboard should show **API connected** when the Colab runtime and tunnel are live.

If Colab disconnects or resets, rerun the notebook cells and update the URL/key in `.env`. The free Colab runtime is temporary and may impose usage limits. Do not use this quick tunnel as a production hosting setup.
